In [1]:
from backend.enrichment import (
    google_places, Config, web_searches, 
    PLACES_API_KEY, OPENAI_API_KEY, TAVILY_API_KEY
)

/Users/riaverma/Library/Mobile Documents/com~apple~CloudDocs/GitHub_Repos/coffee_app/backend/.venv/lib/python3.11/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/riaverma/Library/Mobile Documents/com~apple~CloudDocs/GitHub_Repos/coffee_app/backend/.venv/lib/python3.11/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [3]:
cfg = Config(
    api_key=PLACES_API_KEY,
    lat=40.76894303040112,
    lng=-73.98811816889815,
    radius_m=1200,
    grid_step_m=400,
    nearby_search_radius_m=300,
    include_types=("cafe", "restaurant", "library", "bakery", "lodging"),
    keyword=None,
    max_places_to_enrich=3,      
    request_sleep_s=0.25    
)

In [5]:
d = google_places.nearby_search(cfg, place_type="cafe")

In [4]:
birch_coffee_id = "ChIJhz0V-lhYwokR-0fYpHMII94"

In [4]:
birch_details = google_places.place_details(cfg, birch_coffee_id)

In [5]:
# Test nearby_search with debug
results = google_places.nearby_search(cfg, "cafe", debug=True)

# Test place_details with debug  
details = google_places.place_details(cfg, birch_coffee_id, debug=True)


DEBUG: [nearby_search] Searching for type: cafe
DEBUG: [nearby_search] Location: (40.76894303040112, -73.98811816889815), Radius: 300
DEBUG: [nearby_search] Found 14 results on this page
DEBUG: [nearby_search] First result keys: ['id', 'types', 'formattedAddress', 'location', 'rating', 'websiteUri', 'regularOpeningHours', 'businessStatus', 'priceLevel', 'userRatingCount', 'displayName', 'servesCoffee', 'restroom', 'parkingOptions', 'accessibilityOptions']
DEBUG: [nearby_search] First result id: ChIJ1bM7WeVZwokRdnontikwFK4
DEBUG: [nearby_search] First result displayName: The Oasis Cafe

DEBUG: [nearby_search] Binary attributes in first result:
  - restroom: True (type: bool)
  - servesCoffee: True (type: bool)
  - goodForGroups: None (type: NoneType)
  - parkingOptions: {'paidParkingLot': True, 'paidStreetParking': True} (type: dict)
  - accessibilityOptions: {'wheelchairAccessibleParking': False, 'wheelchairAccessibleEntrance': True} (type: dict)
  - outdoorSeating: None (type: NoneTy

In [10]:
birch_details

{'business_status': 'OPERATIONAL',
 'formatted_address': '884 9th Ave, New York, NY 10019, USA',
 'geometry': {'location': {'lat': 40.7681474, 'lng': -73.98530439999999}},
 'name': 'Birch Coffee',
 'opening_hours': {'open_now': False,
  'periods': [{'close': {'day': 0, 'time': '1600'},
    'open': {'day': 0, 'time': '0730'}},
   {'close': {'day': 1, 'time': '1600'}, 'open': {'day': 1, 'time': '0730'}},
   {'close': {'day': 2, 'time': '1600'}, 'open': {'day': 2, 'time': '0730'}},
   {'close': {'day': 3, 'time': '1600'}, 'open': {'day': 3, 'time': '0730'}},
   {'close': {'day': 4, 'time': '1600'}, 'open': {'day': 4, 'time': '0730'}},
   {'close': {'day': 5, 'time': '1600'}, 'open': {'day': 5, 'time': '0730'}},
   {'close': {'day': 6, 'time': '1600'}, 'open': {'day': 6, 'time': '0730'}}],
  'weekday_text': ['Monday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Tuesday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Wednesday: 7:30\u202fAM\u2009–\u20094:00\u202fPM',
   'Thursday: 7:30\u202fAM\

In [5]:
res = web_searches.enrich_place_with_agent(place=birch_details,
place_id=birch_details["place_id"])

print("Attributes:", res.attributes.model_dump(),
"\n")
print("Evidence:")
for ev in res.evidence:
    print(f"- {ev.url}\n  title={ev.title}\n snippet={ev.snippet}\n  quote={ev.quote}\n")
#print("Raw agent JSON (truncated):", res.raw_agent_output[:1200])

Attributes: {'wifi_quality': 'mixed', 'outlet_availability': 'few', 'noise_level': 'mixed', 'laptop_friendly': 'no', 'time_pressure': 'some', 'seating_comfort': 'mixed', 'best_times': ['weekday mornings', 'mid-afternoon (before closing)'], 'common_complaints': ['very small / cramped space', 'limited or no seating', 'can get busy during peak times'], 'notable_positives': ['high-quality coffee', 'friendly, helpful staff', 'convenient grab-and-go for quick visits'], 'wfh_overall': 'no', 'confidence': 0.45, 'summary': 'Great coffee and staff but small, grab-and-go; limited seating/outlets — not ideal for extended laptop work.'} 

Evidence:
- https://m.yelp.com/biz/birch-coffee-new-york-16?start=40
  title=BIRCH COFFEE - Yelp
 snippet=Small shop with just standing room only so mainly for to-go orders. ... It's a take out place with no seating ... Good place to grab and go for a cup of coffee
  quote=Small shop with just standing room only so mainly for to-go orders.

- https://www.tripadvis

In [21]:
# Test the new enrichment system
# Import new modules
from backend.enrichment.json_storage import load_places_json, upsert_place_ids
from backend.enrichment.places_manager import process_nearby_search_sync, process_enrichment_async, get_places_needing_enrichment
from backend.enrichment.google_places import nearby_search_with_sync
from backend.enrichment.place_enrichment import enrich_place_details_sync, enrich_place_web_async

# Set JSON path
JSON_PATH = "backend/data/places_bootstrap.json"

print("New enrichment system imported successfully!")


New enrichment system imported successfully!


In [22]:
from backend.enrichment import reset_enrichment_flag, process_enrichment_async, Config

# 1. Reset ALL places (keep existing enrichment data for reference)
reset_count = reset_enrichment_flag("backend/data/places_bootstrap.json")
print(f"Reset {reset_count} places")

# # 2. Reset specific places
# reset_count = reset_enrichment_flag(
#     "backend/data/places_bootstrap.json",
#     place_ids=["ChIJW6K4fFlYwokRl9znZwmIvEU", "ChIJhz0V-lhYwokR-0fYpHMII94"]
# )

# # 3. Reset and CLEAR enrichment data (start fresh)
# reset_count = reset_enrichment_flag(
#     "backend/data/places_bootstrap.json",
#     clear_enrichment_data=True
# )


DEBUG: File exists, size: 396229 bytes
DEBUG: Read 393581 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: False
DEBUG: Successfully saved 396231 bytes to file
Reset 2 places


In [4]:
test_cfg = Config(
    api_key=PLACES_API_KEY,
    lat=40.76894303040112,
    lng=-73.98811816889815,
    radius_m=500,  # Smaller radius for testing
    grid_step_m=400,
    nearby_search_radius_m=300,
    include_types=("cafe",),
    keyword=None,
    max_places_to_enrich=2,  # Small number for testing
    request_sleep_s=0.25,
)


In [3]:
# Test 1: Sync operations - nearby_search with auto-enrichment
# This will:
# 1. Run nearby_search
# 2. Upsert place_ids to JSON
# 3. Optionally fetch place_details if places_details_flag is false


# Run nearby_search with sync enrichment
results = nearby_search_with_sync(test_cfg, place_type="cafe", auto_enrich_sync=True)

print(f"Found {len(results)} places")
print(f"First place: {results[0].get('name') if results else 'None'}")


NameError: name 'nearby_search_with_sync' is not defined

In [5]:
# Check what was saved to JSON
places = load_places_json(JSON_PATH)
print(f"Total places in JSON: {len(places)}")

# Show first place
if places:
    first_place_id = list(places.keys())[0]
    first_place = places[first_place_id]
    print(f"\nFirst place ID: {first_place_id}")
    print(f"Places details flag: {first_place.get('places_details_flag')}")
    print(f"Enriched flag: {first_place.get('enriched_flag')}")
    print(f"Place name: {first_place.get('place', {}).get('name', 'N/A')}")
    print(f"Has reviews: {len(first_place.get('sources', {}).get('google_reviews', {}).get('reviews', []))} reviews")


DEBUG: File exists, size: 381617 bytes
DEBUG: Read 378979 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Total places in JSON: 14

First place ID: ChIJ1bM7WeVZwokRdnontikwFK4
Places details flag: True
Enriched flag: False
Place name: The Oasis Cafe
Has reviews: 5 reviews


In [23]:
# Test 2: Async operations - Tavily + LLM enrichment
# This will:
# 1. Run Tavily search for places where enriched_flag is false
# 2. Use LLM to derive attributes from Google reviews + Tavily results
# 3. Save derived attributes to JSON

# Set JSON path
JSON_PATH = "backend/data/places_bootstrap.json"

# Get places needing enrichment
needing_enrichment = get_places_needing_enrichment(JSON_PATH)
print(f"Places needing async enrichment: {len(needing_enrichment)}")

# Run async enrichment for first few places (limit for testing)
if needing_enrichment:
    test_place_ids = needing_enrichment[:2]  # Just test with 2 places
    print(f"Running async enrichment for {len(test_place_ids)} places...")
    
    enriched_count = process_enrichment_async(test_cfg, place_ids=test_place_ids, json_path=JSON_PATH)
    print(f"Enriched {enriched_count} places")
else:
    print("No places need enrichment")


DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Places needing async enrichment: 14
Running async enrichment for 2 places...
DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']


LLM returned has_outlets='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
VALIDATION FAILED: seating_comfort='mixed' (conf=0.5) has evidence that doesn't match attribute keywords. Evidence: ["Review: 'the environment blends clean decor with a relaxing vibe'"]. Keywords required: ['comfortable', 'comfort', 'uncomfortable', 'cushion', 'hard', 'soft', 'ergonomic', 'chair', 'seating']. Overriding to 'unknown' with confidence 0.0


DEBUG: [upsert_place] Upserting place: ChIJ1bM7WeVZwokRdnontikwFK4
DEBUG: File exists, size: 396231 bytes
DEBUG: Read 393583 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: [upsert_place] Place already exists
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: True
DEBUG: Successfully saved 397242 bytes to file
DEBUG: [upsert_place] Successfully upserted: ChIJ1bM7WeVZwokRdnontikwFK4


Failed to generate queries with LLM: Expecting value: line 1 column 1 (char 0), using fallback
LLM returned has_wifi='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
LLM returned has_outlets='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0
VALIDATION FAILED: noise_level='mixed' (conf=0.5) has evidence that doesn't match attribute keywords. Evidence: ["Review 1: 'too many dogs inside this small coffee shop'", "Review 4: 'The atmosphere of this location is second to none.'"]. Keywords required: ['noise', 'noisy', 'quiet', 'quietly', 'loud', 'peaceful', 'calm', 'busy', 'crowded', 'music', 'sound']. Overriding to 'unknown' with confidence 0.0
LLM returned seating_comfort='unknown' with confidence 0.0 but no evidence. Overriding to 'unknown' with confidence 0.0


DEBUG: [upsert_place] Upserting place: ChIJW6K4fFlYwokRl9znZwmIvEU
DEBUG: File exists, size: 397242 bytes
DEBUG: Read 394594 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: [upsert_place] Place already exists
DEBUG: Saving 14 places to JSON file: backend/data/places_bootstrap.json
DEBUG: Sample place_ids being saved: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
DEBUG: First place structure keys: ['place_id', 'places_details_flag', 'enriched_flag', 'place', 'sources', 'derived', 'places_details_called_at', 'places_details_called_version', 'enriched_at', 'enriched_version']
DEBUG: First place flags - places_details: True, enriched: True
DEBUG: Successfully saved 397298 bytes to file
DEBUG: [upsert_place] Successfully upserted: ChIJW6K4fFlYwokRl9znZwmIvEU
E

In [24]:
# Verify async enrichment results
places = load_places_json(JSON_PATH)

# Check if any places have been enriched
enriched_places = [pid for pid, place in places.items() if place.get('enriched_flag')]

if enriched_places:
    test_place_id = enriched_places[0]
    test_place = places[test_place_id]
    
    print(f"Enriched place: {test_place.get('place', {}).get('name')}")
    print(f"\nTavily sources:")
    tavily = test_place.get('sources', {}).get('tavily', {})
    print(f"  Query: {tavily.get('query')}")
    print(f"  Results: {len(tavily.get('results', []))} results")
    print(f"  Excerpts: {len(tavily.get('excerpts', []))} excerpts")
    
    print(f"\nDerived attributes:")
    derived = test_place.get('derived', {})
    for attr_name, attr_data in derived.items():
        if isinstance(attr_data, dict):
            value = attr_data.get('value', 'N/A')
            confidence = attr_data.get('confidence', 0.0)
            sources_count = len(attr_data.get('sources', []))
            evidence_count = len(attr_data.get('evidence', []))
            print(f"  {attr_name}: {value} (confidence: {confidence:.2f}, sources: {sources_count}, evidence: {evidence_count})")
else:
    print("No places have been enriched yet")


DEBUG: File exists, size: 397298 bytes
DEBUG: Read 394650 characters from file
DEBUG: Parsed JSON successfully, type: <class 'dict'>
DEBUG: Loaded 14 places from JSON
DEBUG: Sample place_ids: ['ChIJ1bM7WeVZwokRdnontikwFK4', 'ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
Enriched place: The Oasis Cafe

Tavily sources:
  Query: The Oasis Cafe 857 9th Ave remote work wifi power outlets reviews
  Results: 8 results
  Excerpts: 8 excerpts

Derived attributes:
  has_wifi: free (confidence: 0.70, sources: 1, evidence: 1)
  has_outlets: unknown (confidence: 0.00, sources: 0, evidence: 0)
  is_laptop_friendly: mixed (confidence: 0.50, sources: 1, evidence: 1)
  noise_level: mixed (confidence: 0.50, sources: 1, evidence: 1)
  seating_availability: limited (confidence: 0.70, sources: 1, evidence: 1)
  seating_comfort: unknown (confidence: 0.00, sources: 0, evidence: 0)
  notable_positives: ['Delicious desserts', 'Unique pastries', 'Friendly staff', 'Cozy atmosphere', 'Good coffee']

In [2]:
# ============================================================================
# Tests for New Restructured Enrichment System
# ============================================================================

print("=" * 70)
print("Testing New Restructured Enrichment System")
print("=" * 70)


Testing New Restructured Enrichment System


In [5]:
# Test 1: Basic Info Extraction from nearby_search
from backend.enrichment.places_manager import extract_basic_info_from_nearby_search

# Get a sample nearby_search result
test_results = google_places.nearby_search(test_cfg, "cafe")
if test_results:
    sample_result = test_results[0]
    print("Test 1: Basic Info Extraction")
    print("-" * 70)
    print(f"Sample nearby_search result keys: {list(sample_result.keys())[:10]}...")
    
    # Extract basic info
    basic_info = extract_basic_info_from_nearby_search(sample_result, PLACES_API_KEY)
    
    print(f"\nExtracted basic info keys: {list(basic_info.keys())}")
    print(f"Name: {basic_info.get('name')}")
    print(f"Rating: {basic_info.get('rating')}")
    print(f"User ratings total: {basic_info.get('user_ratings_total')}")
    print(f"Types: {basic_info.get('types', [])[:3]}")
    print(f"Photos count: {len(basic_info.get('photos', []))}")
    print(f"Has opening_hours: {basic_info.get('opening_hours') is not None}")
    print(f"Has formatted_address: {bool(basic_info.get('formatted_address'))}")
    
    # Verify binary attributes are NOT included
    assert 'restroom' not in basic_info, "restroom should not be in basic info"
    assert 'servesCoffee' not in basic_info, "servesCoffee should not be in basic info"
    assert 'outdoorSeating' not in basic_info, "outdoorSeating should not be in basic info"
    print("\n✓ PASS: Binary attributes correctly excluded from basic info")
    
    print("\n" + "=" * 70)


Test 1: Basic Info Extraction
----------------------------------------------------------------------
Sample nearby_search result keys: ['id', 'types', 'formattedAddress', 'location', 'rating', 'websiteUri', 'regularOpeningHours', 'businessStatus', 'priceLevel', 'userRatingCount']...

Extracted basic info keys: ['name', 'lat', 'lng', 'types', 'formatted_address', 'neighborhood', 'website', 'rating', 'user_ratings_total', 'price_level', 'business_status', 'opening_hours', 'photos']
Name: The Oasis Cafe
Rating: 4.4
User ratings total: 370
Types: ['bakery', 'coffee_shop', 'cafe']
Photos count: 5
Has opening_hours: True
Has formatted_address: True

✓ PASS: Binary attributes correctly excluded from basic info



In [7]:
# Test 2: Save Basic Info to JSON
from backend.enrichment.places_manager import save_basic_info_to_json
from backend.enrichment.json_storage import load_places_json

# Use a test JSON path to avoid modifying main file
TEST_JSON_PATH = "backend/data/places_bootstrap_test.json"
import os
if os.path.exists(TEST_JSON_PATH):
    os.remove(TEST_JSON_PATH)

print("Test 2: Save Basic Info to JSON")
print("-" * 70)

# Get nearby_search results
test_results = google_places.nearby_search(test_cfg, "cafe")
if test_results:
    # Limit to 3 for testing
    test_results = test_results[:3]
    print(f"Saving basic info for {len(test_results)} places...")
    
    # Save basic info
    saved_place_ids = save_basic_info_to_json(
        test_results,
        PLACES_API_KEY,
        json_path=TEST_JSON_PATH,
    )
    
    print(f"Saved {len(saved_place_ids)} places")
    
    # Verify saved data
    places = load_places_json(TEST_JSON_PATH)
    print(f"Total places in JSON: {len(places)}")
    
    if saved_place_ids:
        test_place_id = saved_place_ids[0]
        test_place = places[test_place_id]
        
        print(f"\nVerifying place: {test_place_id}")
        print(f"  nearby_search_flag: {test_place.get('nearby_search_flag')}")
        print(f"  places_details_flag: {test_place.get('places_details_flag')}")
        print(f"  enriched_flag: {test_place.get('enriched_flag')}")
        
        place_obj = test_place.get('place', {})
        print(f"  Place name: {place_obj.get('name')}")
        print(f"  Has rating: {place_obj.get('rating') is not None}")
        print(f"  Has photos: {len(place_obj.get('photos', [])) > 0}")
        
        # Verify flags
        assert test_place.get('nearby_search_flag') == True, "nearby_search_flag should be True"
        assert test_place.get('places_details_flag') == False, "places_details_flag should be False"
        assert test_place.get('enriched_flag') == False, "enriched_flag should be False"
        assert place_obj.get('name') is not None, "Place should have name"
        print("\n✓ PASS: Basic info saved correctly with proper flags")
    
    # # Cleanup
    # if os.path.exists(TEST_JSON_PATH):
    #     os.remove(TEST_JSON_PATH)
    #     print(f"\nCleaned up test file: {TEST_JSON_PATH}")

print("\n" + "=" * 70)


Test 2: Save Basic Info to JSON
----------------------------------------------------------------------
Saving basic info for 3 places...


[load_places_json] Timeout acquiring lock for backend/data/places_bootstrap_test.json


Saved 3 places
Total places in JSON: 3

Verifying place: ChIJ1bM7WeVZwokRdnontikwFK4
  nearby_search_flag: True
  places_details_flag: False
  enriched_flag: False
  Place name: The Oasis Cafe
  Has rating: True
  Has photos: True

✓ PASS: Basic info saved correctly with proper flags



In [8]:
# Test 3: Scoring Functions
from backend.enrichment.top_n_scoring import (
    rating_score,
    popularity_score,
    distance_score,
    wfh_type_score,
    score_place,
)

print("Test 3: Scoring Functions")
print("-" * 70)

# Test rating_score
print("rating_score tests:")
assert rating_score(3.5) == 0.0, "3.5 should score 0.0"
assert rating_score(5.0) == 1.0, "5.0 should score 1.0"
assert rating_score(4.25) == 0.5, "4.25 should score 0.5"
assert rating_score(2.0) == 0.0, "2.0 should be clamped to 0.0"
assert rating_score(6.0) == 1.0, "6.0 should be clamped to 1.0"
print("  ✓ rating_score: PASS")

# Test popularity_score
print("\npopularity_score tests:")
max_reviews = 1000
assert popularity_score(0, max_reviews) == 0.0, "0 reviews should score 0.0"
assert popularity_score(100, max_reviews) < popularity_score(1000, max_reviews), "More reviews should score higher"
print(f"  popularity_score(100, {max_reviews}): {popularity_score(100, max_reviews):.3f}")
print(f"  popularity_score(1000, {max_reviews}): {popularity_score(1000, max_reviews):.3f}")
print("  ✓ popularity_score: PASS")

# Test distance_score
print("\ndistance_score tests:")
max_radius = 1000
assert distance_score(0, max_radius) == 1.0, "0m distance should score 1.0"
assert distance_score(max_radius, max_radius) == 0.0, "max_radius distance should score 0.0"
assert distance_score(500, max_radius) == 0.5, "half distance should score 0.5"
print(f"  distance_score(0, {max_radius}): {distance_score(0, max_radius):.3f}")
print(f"  distance_score(500, {max_radius}): {distance_score(500, max_radius):.3f}")
print(f"  distance_score({max_radius}, {max_radius}): {distance_score(max_radius, max_radius):.3f}")
print("  ✓ distance_score: PASS")

# Test wfh_type_score
print("\nwfh_type_score tests:")
assert wfh_type_score(["cafe"]) == 1.0, "cafe should score 1.0"
assert wfh_type_score(["library"]) == 1.0, "library should score 1.0"
assert wfh_type_score(["restaurant"]) == 0.4, "restaurant should score 0.4"
assert wfh_type_score(["night_club"]) == 0.0, "night_club should score 0.0"
assert wfh_type_score(["unknown_type"]) == 0.3, "unknown type should score 0.3"
assert wfh_type_score([]) == 0.2, "empty types should score 0.2"
print("  ✓ wfh_type_score: PASS")

# Test score_place
print("\nscore_place tests:")
test_place = {
    "place": {
        "name": "Test Cafe",
        "lat": 40.76894303040112,
        "lng": -73.98811816889815,
        "rating": 4.5,
        "user_ratings_total": 500,
        "types": ["cafe", "coffee_shop"],
    }
}
user_lat = 40.76894303040112
user_lng = -73.98811816889815
max_radius = 1000
max_reviews = 1000

score = score_place(test_place, user_lat, user_lng, max_radius, max_reviews)
print(f"  score_place result: {score:.3f}")
assert 0.0 <= score <= 1.0, "Score should be between 0 and 1"
print("  ✓ score_place: PASS")

print("\n" + "=" * 70)


Test 3: Scoring Functions
----------------------------------------------------------------------
rating_score tests:
  ✓ rating_score: PASS

popularity_score tests:
  popularity_score(100, 1000): 0.668
  popularity_score(1000, 1000): 1.000
  ✓ popularity_score: PASS

distance_score tests:
  distance_score(0, 1000): 1.000
  distance_score(500, 1000): 0.500
  distance_score(1000, 1000): 0.000
  ✓ distance_score: PASS

wfh_type_score tests:
  ✓ wfh_type_score: PASS

score_place tests:
  score_place result: 0.880
  ✓ score_place: PASS



In [9]:
# Test 4: Top-N Selection
from backend.enrichment.places_manager import select_top_n_places

print("Test 4: Top-N Selection")
print("-" * 70)

# Create test places with different scores
test_places = {}
test_place_ids = []

# High scoring place (cafe, good rating, close)
test_places["place1"] = {
    "place_id": "place1",
    "place": {
        "name": "Great Cafe",
        "lat": 40.76894303040112,
        "lng": -73.98811816889815,
        "rating": 4.8,
        "user_ratings_total": 500,
        "types": ["cafe", "coffee_shop"],
    },
    "enriched_flag": False,
}

# Medium scoring place (restaurant, decent rating)
test_places["place2"] = {
    "place_id": "place2",
    "place": {
        "name": "OK Restaurant",
        "lat": 40.76994303040112,  # Slightly further
        "lng": -73.98911816889815,
        "rating": 4.0,
        "user_ratings_total": 200,
        "types": ["restaurant"],
    },
    "enriched_flag": False,
}

# Low scoring place (bar, low rating)
test_places["place3"] = {
    "place_id": "place3",
    "place": {
        "name": "Noisy Bar",
        "lat": 40.77094303040112,  # Further
        "lng": -73.99011816889815,
        "rating": 3.2,
        "user_ratings_total": 50,
        "types": ["bar"],
    },
    "enriched_flag": False,
}

# Already enriched place (should be filtered out)
test_places["place4"] = {
    "place_id": "place4",
    "place": {
        "name": "Already Enriched",
        "lat": 40.76894303040112,
        "lng": -73.98811816889815,
        "rating": 5.0,
        "user_ratings_total": 1000,
        "types": ["cafe"],
    },
    "enriched_flag": True,  # Already enriched
}

test_place_ids = ["place1", "place2", "place3", "place4"]

# Select top-2
user_lat = 40.76894303040112
user_lng = -73.98811816889815
max_radius = 1000

top_n = select_top_n_places(
    test_places,
    test_place_ids,
    user_lat,
    user_lng,
    max_radius,
    n=2,
)

print(f"Selected top-2 place_ids: {top_n}")
print(f"Expected: place1 should be first (best score)")
print(f"Expected: place2 should be second (better than place3)")
print(f"Expected: place4 should be filtered out (already enriched)")

assert "place1" in top_n, "place1 should be in top-2"
assert "place4" not in top_n, "place4 should be filtered out (already enriched)"
assert len(top_n) <= 2, "Should return at most 2 places"

print("\n✓ PASS: Top-N selection works correctly")

print("\n" + "=" * 70)


Test 4: Top-N Selection
----------------------------------------------------------------------
Selected top-2 place_ids: ['place1', 'place2']
Expected: place1 should be first (best score)
Expected: place2 should be second (better than place3)
Expected: place4 should be filtered out (already enriched)

✓ PASS: Top-N selection works correctly



In [10]:
# Test 5: Place Details Reuses Basic Info
from backend.enrichment.place_enrichment import enrich_place_details_sync

print("Test 5: Place Details Reuses Basic Info")
print("-" * 70)

# Create a test place with basic info already saved
TEST_JSON_PATH = "backend/data/places_bootstrap_test.json"
if os.path.exists(TEST_JSON_PATH):
    os.remove(TEST_JSON_PATH)

# First, save basic info
test_results = google_places.nearby_search(test_cfg, "cafe")
if test_results:
    sample_result = test_results[0]
    place_id = sample_result.get("id")
    
    if place_id:
        # Save basic info
        save_basic_info_to_json([sample_result], PLACES_API_KEY, TEST_JSON_PATH)
        
        # Load the place
        places = load_places_json(TEST_JSON_PATH)
        existing_place = places[place_id]
        
        print(f"Place ID: {place_id}")
        print(f"Before enrichment:")
        print(f"  nearby_search_flag: {existing_place.get('nearby_search_flag')}")
        print(f"  places_details_flag: {existing_place.get('places_details_flag')}")
        print(f"  Place name: {existing_place.get('place', {}).get('name')}")
        print(f"  Has photos: {len(existing_place.get('place', {}).get('photos', [])) > 0}")
        
        # Store original basic info
        original_name = existing_place.get('place', {}).get('name')
        original_photos_count = len(existing_place.get('place', {}).get('photos', []))
        
        # Now enrich with place_details
        print(f"\nRunning place_details enrichment...")
        updated_place = enrich_place_details_sync(
            test_cfg,
            place_id,
            existing_place,
            nearby_result=sample_result,
        )
        
        # Save updated place
        from backend.enrichment.json_storage import upsert_place
        upsert_place(TEST_JSON_PATH, updated_place)
        
        # Verify
        places = load_places_json(TEST_JSON_PATH)
        final_place = places[place_id]
        
        print(f"\nAfter enrichment:")
        print(f"  places_details_flag: {final_place.get('places_details_flag')}")
        print(f"  Place name: {final_place.get('place', {}).get('name')}")
        print(f"  Has photos: {len(final_place.get('place', {}).get('photos', [])) > 0}")
        print(f"  Has reviews: {len(final_place.get('sources', {}).get('google_reviews', {}).get('reviews', [])) > 0}")
        print(f"  Has binary attrs: restroom={final_place.get('place', {}).get('restroom')}")
        
        # Verify basic info was preserved
        assert final_place.get('place', {}).get('name') == original_name, "Name should be preserved"
        assert len(final_place.get('place', {}).get('photos', [])) == original_photos_count, "Photos should be preserved"
        assert final_place.get('places_details_flag') == True, "places_details_flag should be True"
        assert len(final_place.get('sources', {}).get('google_reviews', {}).get('reviews', [])) > 0, "Should have reviews"
        assert final_place.get('place', {}).get('restroom') is not None, "Should have binary attributes"
        
        print("\n✓ PASS: Basic info preserved, reviews and binary attrs added")
    
    # # Cleanup
    # if os.path.exists(TEST_JSON_PATH):
    #     os.remove(TEST_JSON_PATH)

print("\n" + "=" * 70)


Test 5: Place Details Reuses Basic Info
----------------------------------------------------------------------


[load_places_json] Timeout acquiring lock for backend/data/places_bootstrap_test.json


Place ID: ChIJ1bM7WeVZwokRdnontikwFK4
Before enrichment:
  nearby_search_flag: True
  places_details_flag: False
  Place name: The Oasis Cafe
  Has photos: True

Running place_details enrichment...

DEBUG: [enrich_place_details_sync] ============================================================
DEBUG: [enrich_place_details_sync] Extracting binary attributes for ChIJ1bM7WeVZwokRdnontikwFK4
DEBUG: [enrich_place_details_sync] ============================================================
DEBUG: [enrich_place_details_sync] ✓ Using nearby_search result
DEBUG: [enrich_place_details_sync] Nearby result keys: ['id', 'types', 'formattedAddress', 'location', 'rating', 'websiteUri', 'regularOpeningHours', 'businessStatus', 'priceLevel', 'userRatingCount', 'displayName', 'photos', 'servesCoffee', 'restroom', 'parkingOptions', 'accessibilityOptions']

DEBUG: [enrich_place_details_sync] Checking binary attributes in nearby_search:
  ✓ restroom: True (type: bool)
  ✓ servesCoffee: True (type: bool)
  ✗ 

In [12]:
# Test 6: Partial Evidence Handling
from backend.enrichment.place_enrichment import enrich_place_web_async

print("Test 6: Partial Evidence Handling")
print("-" * 70)

# This test verifies that enrich_place_web_async handles cases where
# one API (place_details or Tavily) fails but the other succeeds

# Note: This is a conceptual test - actual API failures are hard to simulate
# But we can verify the logic handles missing data correctly

print("Testing enrich_place_web_async with existing place_details data...")
TEST_JSON_PATH = "backend/data/places_bootstrap_test.json"

# Get a place that has place_details but not Tavily
places = load_places_json(TEST_JSON_PATH)
if places:
    # Find a place with place_details_flag=True but enriched_flag=False
    test_place_id = None
    for pid, place in places.items():
        if place.get('places_details_flag') and not place.get('enriched_flag'):
            test_place_id = pid
            break
    
    if test_place_id:
        test_place = places[test_place_id]
        place_obj = test_place.get('place', {})
        
        print(f"Testing with place: {place_obj.get('name')}")
        print(f"  places_details_flag: {test_place.get('places_details_flag')}")
        print(f"  Has reviews: {len(test_place.get('sources', {}).get('google_reviews', {}).get('reviews', [])) > 0}")
        print(f"  tavily_flag: {test_place.get('tavily_flag', False)}")
        print(f"  enriched_flag: {test_place.get('enriched_flag', False)}")
        
        # Run async enrichment (this will run Tavily + LLM)
        # Note: This will make actual API calls, so it may take time
        print(f"\nRunning async enrichment (Tavily + LLM)...")
        print(f"(This may take 30-60 seconds)")
        
        try:
            updated_place = enrich_place_web_async(test_cfg, place_obj, test_place)
            
            print(f"\nAfter enrichment:")
            print(f"  places_details_flag: {updated_place.get('places_details_flag')}")
            print(f"  tavily_flag: {updated_place.get('tavily_flag')}")
            print(f"  enriched_flag: {updated_place.get('enriched_flag')}")
            print(f"  Has derived attrs: {bool(updated_place.get('derived'))}")
            
            # Verify flags are set correctly
            if updated_place.get('tavily_flag'):
                assert updated_place.get('enriched_flag') == True, "If Tavily succeeds, enriched_flag should be True"
            elif updated_place.get('places_details_flag'):
                # If only place_details has data, enriched_flag might still be True if LLM can use reviews
                print(f"  Note: Only place_details data available, enriched_flag={updated_place.get('enriched_flag')}")
            
            print("\n✓ PASS: Partial evidence handling works")
        except Exception as e:
            print(f"\n⚠ Note: Enrichment failed (this is OK for testing): {e}")
            print("  The function should handle errors gracefully")
    else:
        print("No suitable test place found (need place with place_details but not enriched)")

print("\n" + "=" * 70)


Test 6: Partial Evidence Handling
----------------------------------------------------------------------
Testing enrich_place_web_async with existing place_details data...
Testing with place: The Oasis Cafe
  places_details_flag: True
  Has reviews: True
  tavily_flag: False
  enriched_flag: False

Running async enrichment (Tavily + LLM)...
(This may take 30-60 seconds)


Tavily search failed for query 'The Oasis Cafe 857 9th Ave New York remote work wifi power outlets reviews': This request exceeds your plan's set usage limit. Please upgrade your plan or contact support@tavily.com
Tavily search failed for query 'Is The Oasis Cafe good for working on a laptop seating comfort noise level': This request exceeds your plan's set usage limit. Please upgrade your plan or contact support@tavily.com
Tavily search failed for query 'The Oasis Cafe coworking laptop-friendly environment reviews': This request exceeds your plan's set usage limit. Please upgrade your plan or contact support@tavily.com
Tavily search returned empty results for all queries: ['The Oasis Cafe 857 9th Ave New York remote work wifi power outlets reviews', 'Is The Oasis Cafe good for working on a laptop seating comfort noise level', 'The Oasis Cafe coworking laptop-friendly environment reviews']
Place ChIJ1bM7WeVZwokRdnontikwFK4 Tavily search failed/returned empty, no existing data to preser


After enrichment:
  places_details_flag: True
  tavily_flag: False
  enriched_flag: True
  Has derived attrs: True
  Note: Only place_details data available, enriched_flag=True

✓ PASS: Partial evidence handling works



In [14]:
# Test 7: Verify nearby_search_flag in JSON Schema
print("Test 7: Verify nearby_search_flag in JSON Schema")
print("-" * 70)

places = load_places_json(TEST_JSON_PATH)
if places:
    # Check if places have nearby_search_flag
    places_with_flag = [pid for pid, place in places.items() if 'nearby_search_flag' in place]
    places_without_flag = [pid for pid, place in places.items() if 'nearby_search_flag' not in place]
    
    print(f"Places with nearby_search_flag: {len(places_with_flag)}")
    print(f"Places without nearby_search_flag: {len(places_without_flag)}")
    
    if places_with_flag:
        sample_place_id = places_with_flag[0]
        sample_place = places[sample_place_id]
        print(f"\nSample place flags:")
        print(f"  nearby_search_flag: {sample_place.get('nearby_search_flag')}")
        print(f"  places_details_flag: {sample_place.get('places_details_flag')}")
        print(f"  tavily_flag: {sample_place.get('tavily_flag')}")
        print(f"  enriched_flag: {sample_place.get('enriched_flag')}")
    
    print("\n✓ PASS: nearby_search_flag exists in JSON schema")

print("\n" + "=" * 70)


Test 7: Verify nearby_search_flag in JSON Schema
----------------------------------------------------------------------
Places with nearby_search_flag: 1
Places without nearby_search_flag: 0

Sample place flags:
  nearby_search_flag: True
  places_details_flag: True
  tavily_flag: False
  enriched_flag: False

✓ PASS: nearby_search_flag exists in JSON schema



In [15]:
# Test 8: End-to-End Flow Test
# This simulates the new flow:
# 1. nearby_search returns results
# 2. Basic info is saved immediately
# 3. Results are returned immediately (without waiting for enrichment)

print("Test 8: End-to-End Flow Test")
print("-" * 70)

TEST_JSON_PATH = "backend/data/places_bootstrap_test.json"
if os.path.exists(TEST_JSON_PATH):
    os.remove(TEST_JSON_PATH)

# Step 1: Run nearby_search
print("Step 1: Running nearby_search...")
test_results = google_places.nearby_search(test_cfg, "cafe")
print(f"  Found {len(test_results)} results")

if test_results:
    # Limit to 3 for testing
    test_results = test_results[:3]
    
    # Step 2: Save basic info immediately
    print("\nStep 2: Saving basic info immediately...")
    saved_place_ids = save_basic_info_to_json(
        test_results,
        PLACES_API_KEY,
        json_path=TEST_JSON_PATH,
    )
    print(f"  Saved {len(saved_place_ids)} places")
    
    # Step 3: Verify we can return basic info immediately
    print("\nStep 3: Verifying basic info is available immediately...")
    places = load_places_json(TEST_JSON_PATH)
    
    for place_id in saved_place_ids[:2]:  # Check first 2
        place = places[place_id]
        place_obj = place.get('place', {})
        
        print(f"\n  Place: {place_obj.get('name')}")
        print(f"    nearby_search_flag: {place.get('nearby_search_flag')}")
        print(f"    Has name: {bool(place_obj.get('name'))}")
        print(f"    Has rating: {place_obj.get('rating') is not None}")
        print(f"    Has photos: {len(place_obj.get('photos', [])) > 0}")
        print(f"    Has types: {len(place_obj.get('types', [])) > 0}")
        
        # Verify basic info is present
        assert place.get('nearby_search_flag') == True, "Should have nearby_search_flag"
        assert place_obj.get('name') is not None, "Should have name"
        assert place_obj.get('rating') is not None or place_obj.get('user_ratings_total') is not None, "Should have rating info"
    
    print("\n✓ PASS: Basic info available immediately without enrichment")
    
    # Step 4: Simulate background enrichment (select top-n and enrich)
    print("\nStep 4: Simulating background enrichment (top-n selection)...")
    user_lat = test_cfg.lat
    user_lng = test_cfg.lng
    max_radius = test_cfg.nearby_search_radius_m
    
    top_n = select_top_n_places(
        places,
        saved_place_ids,
        user_lat,
        user_lng,
        max_radius,
        n=2,  # Top 2 for testing
    )
    
    print(f"  Selected top-2 places: {top_n}")
    print(f"  These would be enriched in background task")
    
    print("\n✓ PASS: End-to-end flow works correctly")
    
    # Cleanup
    if os.path.exists(TEST_JSON_PATH):
        os.remove(TEST_JSON_PATH)
        print(f"\nCleaned up test file: {TEST_JSON_PATH}")

print("\n" + "=" * 70)
print("All Tests Complete!")
print("=" * 70)


Test 8: End-to-End Flow Test
----------------------------------------------------------------------
Step 1: Running nearby_search...
  Found 14 results

Step 2: Saving basic info immediately...


[load_places_json] Timeout acquiring lock for backend/data/places_bootstrap_test.json


  Saved 3 places

Step 3: Verifying basic info is available immediately...

  Place: The Oasis Cafe
    nearby_search_flag: True
    Has name: True
    Has rating: True
    Has photos: True
    Has types: True

  Place: Rex
    nearby_search_flag: True
    Has name: True
    Has rating: True
    Has photos: True
    Has types: True

✓ PASS: Basic info available immediately without enrichment

Step 4: Simulating background enrichment (top-n selection)...
  Selected top-2 places: ['ChIJW6K4fFlYwokRl9znZwmIvEU', 'ChIJ1-tsCsFZwokRCUumkRj0B_w']
  These would be enriched in background task

✓ PASS: End-to-end flow works correctly

Cleaned up test file: backend/data/places_bootstrap_test.json

All Tests Complete!
